# Import Data and Setup

In [ ]:
import pandas as pd
import numpy as np
import math
import ast
import os

In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# configure the data path
DRIVE_ROOT = '/content/drive/MyDrive/16th grade Academics/A S26/CSCI 6980'
LOCAL_WORK_PATH = 'content/hockey_processing_folder'
CSV_PATH = os.path.join(DRIVE_ROOT, 'player_locations')

In [ ]:
# import csvs to dfs
GT_CSV = os.path.join(CSV_PATH, 'tom_annotated_coords.csv')
gt_df = pd.read_csv(GT_CSV)

TRACKER_CSV = os.path.join(CSV_PATH, 'tom_annotated_ByteTrack.csv')
tracker_df = pd.read_csv(TRACKER_CSV)

In [ ]:
# edit gt frame column from tuple to number
def extract_frame(ses_frm_string):
    try:
        return ast.literal_eval(ses_frm_string)[1]
    except:
        return -1

gt_df['Frame'] = gt_df['Ses-Frm'].apply(extract_frame)

In [ ]:
print(f'Loaded {len(gt_df)} ground truth frames')
print(f'Loaded {len(tracker_df)} tracker frames')

Loaded 136 ground truth frames
Loaded 2720 tracker frames


# Evaluation

In [ ]:
PIXEL_THRESHOLD = 50

# regulation period
# players_to_evaluate = ['H1', 'H2', 'H3', 'H4', 'H5', 'V1', 'V2', 'V3', 'V4', 'V5']

# overtime period
players_to_evaluate = ['H1', 'H2', 'H3', 'V1', 'V2', 'V3']

results = []

# walk througth all the gt players
for player in players_to_evaluate:
    if player not in gt_df.columns:
        continue

    total_gt_frames = 0
    false_negatives = 0
    id_switches = 0

    tracked_ids = []

    # walk through all the frames for the player
    for index, row in gt_df.iterrows():
        frame_num = row['Frame']
        gt_coord_str = str(row[player])

        # if the player isn't being tracked, skip frame
        if pd.isna(gt_coord_str) or gt_coord_str == "(-1, -1)":
            continue

        gt_point = ast.literal_eval(gt_coord_str)
        total_gt_frames += 1

        frame_detections = tracker_df[tracker_df['Frame'] == frame_num]

        best_match_id = None
        min_dist = float('inf')

        # check all of the boxes to see if there is overlap with gt
        for _, det_row in frame_detections.iterrows():
            trk_point = (det_row['X_Coordinate'], det_row['Y_Coordinate'])
            dist = math.dist(gt_point, trk_point)

            if dist < PIXEL_THRESHOLD and dist < min_dist:
                min_dist = dist
                best_match_id = det_row['ID']

        # scoring logic
        if best_match_id is not None:
            # check to see if the ID has switched
            if len(tracked_ids) > 0 and best_match_id != tracked_ids[-1]:
                id_switches += 1
            tracked_ids.append(best_match_id)
        else:
            false_negatives += 1

    # MOTA for player
    if total_gt_frames > 0:
        # MOTA = 1 - (Errors / Total Annotations)
        mota_score = 1.0 - ((false_negatives + id_switches) / total_gt_frames)
        coverage_pct = (len(tracked_ids) / total_gt_frames) * 100

        results.append({
            'Player': player,
            'Total_Frames': total_gt_frames,
            'Successful_Matches': len(tracked_ids),
            'False_Negatives (Misses)': false_negatives,
            'ID_Switches': id_switches,
            'Coverage_%': round(coverage_pct, 2),
            'Custom_MOTA': round(mota_score, 4)
        })

print("Evaluation Complete!")

Evaluation Complete!


In [ ]:
results_df = pd.DataFrame(results)

# final average score for the whole video
avg_mota = results_df['Custom_MOTA'].mean()
print(f"Overall Domain-Adapted MOTA Score: {avg_mota:.2%}\n")

display(results_df)

Overall Domain-Adapted MOTA Score: 91.86%



,Player,Total_Frames,Successful_Matches,False_Negatives (Misses),ID_Switches,Coverage_%,Custom_MOTA
0,H1,136,129,7,1,94.85,0.9412
1,H2,136,136,0,0,100.00,1.0000
2,H3,136,136,0,4,100.00,0.9706
3,V1,136,136,0,0,100.00,1.0000
4,V2,120,90,30,18,75.00,0.6000
5,V3,74,74,0,0,100.00,1.0000
